In [ ]:
import requests
import pandas as pd

In [ ]:
%pip install requests

In [ ]:
import requests
import pandas as pd

In [ ]:
import requests
import pandas as pd

In [ ]:
import os
client_id = os.getenv("NAVER_CLIENT_ID")
client_secret = os.getenv("NAVER_CLIENT_SECRET")

url = "https://naverapihub.apigw.ntruss.com/search-trend/v1/search"

headers = {
    "X-NCP-APIGW-API-KEY-ID": client_id,
    "X-NCP-APIGW-API-KEY": client_secret,
    "Content-Type": "application/json"
}

In [ ]:
data = {
    "startDate": "2026-01-01",
    "endDate": "2026-08-31",
    "timeUnit": "month",
    "keywordGroups": [
        {
            "groupName": "카페",
            "keywords": ["카페", "커피", "브런치"]
        }
    ],
    "ages": ["7", "8"]
}

response = requests.post(
    url,
    headers=headers,
    json=data
)

print("상태코드:", response.status_code)
print(response.json())

In [ ]:
result = response.json()

trend_df = pd.DataFrame(
    result["results"][0]["data"]
)

trend_df

In [ ]:
shopping_url = "https://naverapihub.apigw.ntruss.com/shopping/v1/category/age"

shopping_data = {
    "startDate": "2026-01-01",
    "endDate": "2026-08-31",
    "timeUnit": "month",
    "category": "50000000"
}

shopping_response = requests.post(
    shopping_url,
    headers=headers,
    json=shopping_data
)

print("상태코드:", shopping_response.status_code)
print(shopping_response.json())

In [ ]:
shopping_response = requests.post(
    shopping_url,
    headers=headers,
    json=shopping_data
)

print("상태코드:", shopping_response.status_code)
print(shopping_response.json())

In [ ]:
shopping_result = shopping_response.json()

shopping_df = pd.DataFrame(
    shopping_result["results"][0]["data"]
)

shopping_df.head(20)

In [ ]:
age_avg = (
    shopping_df
    .groupby("group")["ratio"]
    .mean()
    .reset_index()
)

age_avg["연령대"] = age_avg["group"].map({
    "10": "10대",
    "20": "20대",
    "30": "30대",
    "40": "40대",
    "50": "50대",
    "60": "60대이상"
})

age_avg = age_avg[["연령대", "ratio"]]
age_avg["ratio"] = age_avg["ratio"].round(1)

age_avg

In [ ]:
def get_shopping_age(category_code, start_date="2026-01-01", end_date="2026-08-31"):
    
    shopping_url = "https://naverapihub.apigw.ntruss.com/shopping/v1/category/age"

    data = {
        "startDate": start_date,
        "endDate": end_date,
        "timeUnit": "month",
        "category": category_code
    }

    response = requests.post(
        shopping_url,
        headers=headers,
        json=data
    )

    if response.status_code != 200:
        print("API 오류:", response.status_code)
        print(response.json())
        return None

    result = response.json()

    df = pd.DataFrame(
        result["results"][0]["data"]
    )

    age_avg = (
        df
        .groupby("group")["ratio"]
        .mean()
        .reset_index()
    )

    age_avg["연령대"] = age_avg["group"].map({
        "10": "10대",
        "20": "20대",
        "30": "30대",
        "40": "40대",
        "50": "50대",
        "60": "60대이상"
    })

    age_avg = age_avg[["연령대", "ratio"]]
    age_avg["ratio"] = age_avg["ratio"].round(1)

    return age_avg

In [ ]:
get_shopping_age(
    "50000000",
    start_date="2026-03-01",
    end_date="2026-06-30"
)

In [ ]:
get_shopping_age(
    "50000000",
    start_date="2026-07-01",
    end_date="2026-08-31"
)

In [ ]:
import pandas as pd
import requests

customer_strategy = pd.read_csv(
    "../data/processed/seoul_customer_strategy.csv",
    encoding="utf-8-sig"
)

customer_strategy.head()

In [ ]:
naver_age_map = {
    "20대이하": ["1", "2"],
    "20대": ["3", "4"],
    "30대": ["5", "6"],
    "40대": ["7", "8"],
    "50대": ["9", "10"],
    "60대이상": ["11"]
}

In [ ]:
def get_search_interest(
    age_group,
    keyword_groups,
    start_date="2026-01-01",
    end_date="2026-08-31"
):
    search_url = "https://naverapihub.apigw.ntruss.com/search-trend/v1/search"

    data = {
        "startDate": start_date,
        "endDate": end_date,
        "timeUnit": "month",
        "keywordGroups": [
            {
                "groupName": name,
                "keywords": keywords
            }
            for name, keywords in keyword_groups.items()
        ],
        "ages": naver_age_map[age_group]
    }

    response = requests.post(
        search_url,
        headers=headers,
        json=data
    )

    if response.status_code != 200:
        print("API 오류:", response.status_code)
        print(response.json())
        return None

    rows = []

    for result in response.json()["results"]:
        for item in result["data"]:
            rows.append({
                "주제": result["title"],
                "기간": item["period"],
                "관심도": item["ratio"]
            })

    return pd.DataFrame(rows)

In [ ]:
keywords = {
    "파스타": ["파스타", "스파게티"],
    "스테이크": ["스테이크"],
    "브런치": ["브런치"],
    "샐러드": ["샐러드"],
    "피자": ["피자"]
}

get_search_interest("50대", keywords)

In [ ]:
import os
client_id = os.getenv("NAVER_CLIENT_ID")
client_secret = os.getenv("NAVER_CLIENT_SECRET")

headers = {
    "X-NCP-APIGW-API-KEY-ID": client_id,
    "X-NCP-APIGW-API-KEY": client_secret,
    "Content-Type": "application/json"
}

In [ ]:
get_search_interest("50대", keywords)

In [ ]:
search_50 = get_search_interest(
    "50대",
    keywords
)

search_50.head(20)

In [ ]:
keyword_summary = (
    search_50
    .groupby("주제")["관심도"]
    .mean()
    .round(1)
    .sort_values(ascending=False)
    .reset_index()
)

keyword_summary

In [ ]:
peak_month = (
    search_50.loc[
        search_50.groupby("주제")["관심도"].idxmax(),
        ["주제", "기간", "관심도"]
    ]
    .sort_values("관심도", ascending=False)
    .reset_index(drop=True)
)

peak_month

In [ ]:
def get_keyword_ranking(age_group, keywords):
    df = get_search_interest(age_group, keywords)

    summary = (
        df.groupby("주제")["관심도"]
        .mean()
        .sort_values(ascending=False)
        .reset_index()
    )

    summary["순위"] = range(1, len(summary) + 1)
    summary["연령대"] = age_group

    return summary[["연령대", "주제", "관심도", "순위"]]

In [ ]:
core_rank = get_keyword_ranking("30대", keywords)
next_rank = get_keyword_ranking("50대", keywords)

core_rank

In [ ]:
test_30 = get_search_interest(
    "30대",
    {
        "피자": ["피자"]
    }
)

test_30

In [ ]:
core_rank = get_keyword_ranking("30대", keywords)

In [ ]:
def get_keyword_ranking(age_group, keywords):
    df = get_search_interest(age_group, keywords)

    if df is None:
        print(f"{age_group} 데이터를 가져오지 못했습니다.")
        return None

    summary = (
        df.groupby("주제")["관심도"]
        .mean()
        .sort_values(ascending=False)
        .reset_index()
    )

    summary["순위"] = range(1, len(summary) + 1)
    summary["연령대"] = age_group

    return summary[["연령대", "주제", "관심도", "순위"]]

In [ ]:
core_rank = get_keyword_ranking("30대", keywords)

core_rank

In [ ]:
next_rank = get_keyword_ranking("50대", keywords)

next_rank

In [ ]:
menu_compare = core_rank.merge(
    next_rank,
    on="주제",
    suffixes=("_현재고객", "_NextCustomer")
)

menu_compare["순위변화"] = (
    menu_compare["순위_현재고객"]
    - menu_compare["순위_NextCustomer"]
)

menu_compare = menu_compare[
    [
        "주제",
        "순위_현재고객",
        "순위_NextCustomer",
        "순위변화"
    ]
].sort_values(
    "순위변화",
    ascending=False
)

menu_compare

In [ ]:
menu_recommend = (
    menu_compare[
        menu_compare["순위변화"] > 0
    ]
    .sort_values(
        ["순위_NextCustomer", "순위변화"],
        ascending=[True, False]
    )
    .head(2)
    .reset_index(drop=True)
)

menu_recommend

In [ ]:
recommended_topics = menu_recommend["주제"].tolist()

marketing_timing = (
    search_50[
        search_50["주제"].isin(recommended_topics)
    ]
    .loc[
        lambda x: x.groupby("주제")["관심도"].idxmax(),
        ["주제", "기간", "관심도"]
    ]
    .reset_index(drop=True)
)

marketing_timing

In [ ]:
marketing_solution = pd.DataFrame([{
    "업종명": "서양음식",
    "현재핵심고객": "30대",
    "NextCustomer": "50대",
    "추천메뉴후보": menu_recommend.iloc[0]["주제"],
    "집중시기": marketing_timing.iloc[0]["기간"],
    "검색관심도": round(marketing_timing.iloc[0]["관심도"], 1),
    "추천방향": "Next Customer에서 상대적 관심 순위가 상승한 메뉴를 활용한 프로모션 검토"
}])

marketing_solution

In [ ]:
def make_marketing_solution(
    industry,
    core_age,
    next_age,
    keywords
):
    # 현재 핵심고객과 Next Customer의 키워드 순위
    core_rank = get_keyword_ranking(core_age, keywords)
    next_rank = get_keyword_ranking(next_age, keywords)

    if core_rank is None or next_rank is None:
        return None

    compare = core_rank.merge(
        next_rank,
        on="주제",
        suffixes=("_현재고객", "_NextCustomer")
    )

    compare["순위변화"] = (
        compare["순위_현재고객"]
        - compare["순위_NextCustomer"]
    )

    # Next Customer에서 순위가 상승한 메뉴 중 가장 높은 것
    recommend = (
        compare[compare["순위변화"] > 0]
        .sort_values(
            ["순위_NextCustomer", "순위변화"],
            ascending=[True, False]
        )
    )

    if recommend.empty:
        return pd.DataFrame([{
            "업종명": industry,
            "현재핵심고객": core_age,
            "NextCustomer": next_age,
            "추천메뉴후보": "뚜렷한 후보 없음"
        }])

    best_menu = recommend.iloc[0]["주제"]

    # Next Customer 검색 데이터
    next_search = get_search_interest(next_age, keywords)

    menu_search = next_search[
        next_search["주제"] == best_menu
    ]

    peak = menu_search.loc[
        menu_search["관심도"].idxmax()
    ]

    return pd.DataFrame([{
        "업종명": industry,
        "현재핵심고객": core_age,
        "NextCustomer": next_age,
        "추천메뉴후보": best_menu,
        "집중시기": peak["기간"],
        "검색관심도": round(peak["관심도"], 1),
        "추천방향": "Next Customer에서 상대적 관심 순위가 상승한 메뉴를 활용한 프로모션 검토"
    }])

In [ ]:
make_marketing_solution(
    "서양음식",
    "30대",
    "50대",
    keywords
)

In [ ]:
test_case = customer_solution[
    (customer_solution["업종명"] == "서양음식") &
    (customer_solution["확장형고객"].notna())
].iloc[0]

test_case[
    ["시군구", "업종명", "현재핵심연령대", "확장형고객"]
]

In [ ]:
test_case = customer_strategy[
    (customer_strategy["업종명"] == "서양음식") &
    (customer_strategy["확장형고객"].notna())
].iloc[0]

test_case[
    ["시군구", "업종명", "현재핵심연령대", "확장형고객"]
]

In [ ]:
test_marketing = make_marketing_solution(
    industry=test_case["업종명"],
    core_age=test_case["현재핵심연령대"],
    next_age=test_case["확장형고객"],
    keywords=keywords
)

test_marketing

In [ ]:
convenience_keywords = {
    "도시락": ["편의점 도시락", "도시락"],
    "간편식": ["간편식", "즉석식품"],
    "샌드위치": ["편의점 샌드위치", "샌드위치"],
    "커피": ["편의점 커피", "커피"],
    "건강식품": ["건강식품", "건강 간식"]
}

In [ ]:
test_case_store = customer_strategy[
    (customer_strategy["업종명"] == "편의점") &
    (customer_strategy["확장형고객"].notna())
].iloc[0]

test_case_store[
    ["시군구", "업종명", "현재핵심연령대", "확장형고객"]
]

In [ ]:
store_marketing = make_marketing_solution(
    industry=test_case_store["업종명"],
    core_age=test_case_store["현재핵심연령대"],
    next_age=test_case_store["확장형고객"],
    keywords=convenience_keywords
)

store_marketing

In [ ]:
food_age = get_shopping_age(
    "50000006",
    start_date="2026-01-01",
    end_date="2026-08-31"
)

food_age

In [ ]:
def get_shopping_age(
    category_code,
    start_date="2026-01-01",
    end_date="2026-08-31"
):
    shopping_url = "https://naverapihub.apigw.ntruss.com/shopping/v1/category/age"

    data = {
        "startDate": start_date,
        "endDate": end_date,
        "timeUnit": "month",
        "category": category_code
    }

    response = requests.post(
        shopping_url,
        headers=headers,
        json=data
    )

    if response.status_code != 200:
        print("API 오류:", response.status_code)
        print(response.json())
        return None

    result = response.json()

    df = pd.DataFrame(
        result["results"][0]["data"]
    )

    age_avg = (
        df
        .groupby("group")["ratio"]
        .mean()
        .reset_index()
    )

    age_avg["연령대"] = age_avg["group"].map({
        "10": "10대",
        "20": "20대",
        "30": "30대",
        "40": "40대",
        "50": "50대",
        "60": "60대이상"
    })

    age_avg = age_avg[["연령대", "ratio"]]
    age_avg["ratio"] = age_avg["ratio"].round(1)

    return age_avg

In [ ]:
food_age = get_shopping_age(
    "50000006",
    start_date="2026-01-01",
    end_date="2026-08-31"
)

food_age

In [ ]:
def get_shopping_keyword_age(
    keyword,
    category_code="50000006",
    start_date="2026-01-01",
    end_date="2026-08-31"
):
    url = "https://naverapihub.apigw.ntruss.com/shopping/v1/category/keyword/age"

    data = {
        "startDate": start_date,
        "endDate": end_date,
        "timeUnit": "month",
        "category": category_code,
        "keyword": keyword
    }

    response = requests.post(
        url,
        headers=headers,
        json=data
    )

    if response.status_code != 200:
        print("API 오류:", response.status_code)
        print(response.json())
        return None

    result = response.json()

    df = pd.DataFrame(
        result["results"][0]["data"]
    )

    age_avg = (
        df.groupby("group")["ratio"]
        .mean()
        .reset_index()
    )

    age_avg["연령대"] = age_avg["group"].map({
        "10": "10대",
        "20": "20대",
        "30": "30대",
        "40": "40대",
        "50": "50대",
        "60": "60대이상"
    })

    return age_avg[["연령대", "ratio"]]

In [ ]:
get_shopping_keyword_age("간편식")

In [ ]:
get_shopping_keyword_age("제로음료")
get_shopping_keyword_age("컵라면")
get_shopping_keyword_age("샌드위치")
get_shopping_keyword_age("커피")

In [ ]:
store_keywords = [
    "간편식",
    "제로음료",
    "컵라면",
    "샌드위치",
    "커피"
]

In [ ]:
results = []

for keyword in store_keywords:
    df = get_shopping_keyword_age(keyword)

    if df is None or df.empty:
        continue

    # 해당 키워드 안에서 연령대 순위
    df = df.sort_values("ratio", ascending=False).reset_index(drop=True)
    df["연령순위"] = range(1, len(df) + 1)

    age20 = df[df["연령대"] == "20대"]

    if not age20.empty:
        results.append({
            "상품후보": keyword,
            "20대관심지수": round(age20.iloc[0]["ratio"], 1),
            "20대연령순위": int(age20.iloc[0]["연령순위"])
        })

store_product_candidates = pd.DataFrame(results)

store_product_candidates = store_product_candidates.sort_values(
    ["20대연령순위", "20대관심지수"],
    ascending=[True, False]
).reset_index(drop=True)

store_product_candidates

In [ ]:
store_search_keywords = {
    "제로음료": ["제로음료", "제로콜라"],
    "간편식": ["간편식", "편의점 간편식"],
    "샌드위치": ["샌드위치", "편의점 샌드위치"],
    "컵라면": ["컵라면", "편의점 라면"],
    "커피": ["편의점 커피", "커피"]
}

search_20_store = get_search_interest(
    "20대",
    store_search_keywords
)

store_search_summary = (
    search_20_store
    .groupby("주제")["관심도"]
    .mean()
    .round(1)
    .sort_values(ascending=False)
    .reset_index()
)

store_search_summary

In [ ]:
search_50_store = get_search_interest(
    "50대",
    store_search_keywords
)

store_50_summary = (
    search_50_store
    .groupby("주제")["관심도"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

store_50_summary["50대순위"] = range(
    1, len(store_50_summary) + 1
)

store_20_summary = store_search_summary.copy()

store_20_summary["20대순위"] = range(
    1, len(store_20_summary) + 1
)

In [ ]:
store_compare = store_20_summary[
    ["주제", "20대순위"]
].merge(
    store_50_summary[
        ["주제", "50대순위"]
    ],
    on="주제"
)

store_compare["20대상대상승"] = (
    store_compare["50대순위"]
    - store_compare["20대순위"]
)

store_compare.sort_values(
    "20대상대상승",
    ascending=False
)

In [ ]:
store_search_keywords_v2 = {
    "디저트": ["편의점 디저트", "디저트"],
    "프로틴": ["프로틴", "단백질 음료"],
    "에너지음료": ["에너지드링크", "에너지음료"],
    "도시락": ["편의점 도시락", "도시락"],
    "아이스크림": ["편의점 아이스크림", "아이스크림"]
}

In [ ]:
search_20_v2 = get_search_interest(
    "20대",
    store_search_keywords_v2
)

In [ ]:
search_50_v2 = get_search_interest(
    "50대",
    store_search_keywords_v2
)

In [ ]:
summary_20_v2 = (
    search_20_v2
    .groupby("주제")["관심도"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

summary_20_v2["20대순위"] = range(1, len(summary_20_v2) + 1)


summary_50_v2 = (
    search_50_v2
    .groupby("주제")["관심도"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

summary_50_v2["50대순위"] = range(1, len(summary_50_v2) + 1)

In [ ]:
compare_v2 = summary_20_v2[
    ["주제", "20대순위"]
].merge(
    summary_50_v2[
        ["주제", "50대순위"]
    ],
    on="주제"
)

compare_v2["20대상대상승"] = (
    compare_v2["50대순위"]
    - compare_v2["20대순위"]
)

compare_v2.sort_values(
    "20대상대상승",
    ascending=False
)

In [ ]:
store_recommend = (
    compare_v2[
        compare_v2["20대상대상승"] > 0
    ]
    .sort_values(
        "20대상대상승",
        ascending=False
    )
    .reset_index(drop=True)
)

store_recommend

In [ ]:
recommended_store_topics = store_recommend["주제"].tolist()

store_timing = (
    search_20_v2[
        search_20_v2["주제"].isin(recommended_store_topics)
    ]
    .loc[
        lambda x: x.groupby("주제")["관심도"].idxmax(),
        ["주제", "기간", "관심도"]
    ]
    .reset_index(drop=True)
)

store_timing

In [ ]:
store_marketing_solution = pd.DataFrame([{
    "업종명": "편의점",
    "현재핵심고객": "50대",
    "확장형고객": "20대",
    "추천상품1": store_recommend.iloc[0]["주제"],
    "추천상품2": store_recommend.iloc[1]["주제"],
    "상품1집중시기": store_timing.loc[
        store_timing["주제"] == store_recommend.iloc[0]["주제"],
        "기간"
    ].iloc[0],
    "상품2집중시기": store_timing.loc[
        store_timing["주제"] == store_recommend.iloc[1]["주제"],
        "기간"
    ].iloc[0],
    "추천방향": "확장형 고객에서 상대적 검색 관심 순위가 높은 상품군 중심 프로모션 검토"
}])

store_marketing_solution

In [ ]:
hansik_keywords = {
    "국밥": ["국밥"],
    "찌개": ["김치찌개", "된장찌개", "찌개"],
    "제육": ["제육볶음", "제육"],
    "비빔밥": ["비빔밥"],
    "불고기": ["불고기"]
}

In [ ]:
print("업종별 사례 수")
print(customer_strategy["업종명"].value_counts())

print("\n확장형 고객이 있는 업종")
print(
    customer_strategy[
        customer_strategy["확장형고객"].notna()
    ]["업종명"].value_counts()
)

print("\n방어형 고객이 있는 업종")
print(
    customer_strategy[
        customer_strategy["방어형고객"].notna()
    ]["업종명"].value_counts()
)

In [ ]:
hansik_keywords = {
    "국밥": ["국밥"],
    "찌개": ["김치찌개", "된장찌개", "찌개"],
    "제육": ["제육볶음", "제육"],
    "비빔밥": ["비빔밥"],
    "불고기": ["불고기"]
}

In [ ]:
hansik_case = customer_strategy[
    (customer_strategy["업종명"] == "일반한식") &
    (customer_strategy["확장형고객"].notna())
].iloc[0]

hansik_case[
    ["시군구", "업종명", "현재핵심연령대", "확장형고객"]
]

In [ ]:
hansik_marketing = make_marketing_solution(
    industry=hansik_case["업종명"],
    core_age=hansik_case["현재핵심연령대"],
    next_age=hansik_case["확장형고객"],
    keywords=hansik_keywords
)

hansik_marketing

In [ ]:
startup_candidates = seoul_result[
    (seoul_result["6개월완전"] == True) &
    (seoul_result["2035 순인구효과"] >= 0)
].copy()

startup_candidates = startup_candidates.sort_values(
    ["2035 순인구효과", "월평균증감률"],
    ascending=[False, False]
)

startup_candidates[
    [
        "시군구",
        "업종명",
        "연령대",
        "이용금액비중",
        "연령집중도",
        "월평균증감률",
        "2035 순인구효과",
        "2040 순인구효과"
    ]
].head(20)

In [ ]:
import pandas as pd

analysis_base = pd.read_csv(
    "../data/processed/bc_analysis_base.csv",
    encoding="utf-8-sig"
)

In [ ]:
[x for x in globals() if "search" in x.lower() or "naver" in x.lower()]

In [ ]:
keyword_groups = {
    "파스타": ["파스타"],
    "스테이크": ["스테이크"],
    "브런치": ["브런치"],
    "샐러드": ["샐러드"],
    "피자": ["피자"]
}

test = get_search_interest(
    age_group="20대",
    keyword_groups=keyword_groups
)

test.head()

In [ ]:
import pandas as pd
import requests

In [ ]:
headers
naver_age_map

In [ ]:
import os
import pandas as pd
import requests

# ==========================================
# 1. NAVER API 인증정보
# ==========================================

client_id = os.getenv("NAVER_CLIENT_ID")
client_secret = os.getenv("NAVER_CLIENT_SECRET")

headers = {
    "X-NCP-APIGW-API-KEY-ID": client_id,
    "X-NCP-APIGW-API-KEY": client_secret,
    "Content-Type": "application/json"
}


# ==========================================
# 2. NAVER 연령대 매핑
# ==========================================

naver_age_map = {
    "20대이하": ["1", "2"],
    "20대": ["3", "4"],
    "30대": ["5", "6"],
    "40대": ["7", "8"],
    "50대": ["9", "10"],
    "60대이상": ["11"]
}


# ==========================================
# 3. 검색 관심도 함수
# ==========================================

def get_search_interest(
    age_group,
    keyword_groups,
    start_date="2026-01-01",
    end_date="2026-08-31"
):
    search_url = (
        "https://naverapihub.apigw.ntruss.com/"
        "search-trend/v1/search"
    )

    data = {
        "startDate": start_date,
        "endDate": end_date,
        "timeUnit": "month",

        "keywordGroups": [
            {
                "groupName": name,
                "keywords": keywords
            }
            for name, keywords in keyword_groups.items()
        ],

        "ages": naver_age_map[age_group]
    }

    response = requests.post(
        search_url,
        headers=headers,
        json=data
    )

    if response.status_code != 200:
        print("API 오류:", response.status_code)
        print(response.text)
        return None

    rows = []

    for result in response.json()["results"]:
        for item in result["data"]:
            rows.append({
                "주제": result["title"],
                "기간": item["period"],
                "검색도": item["ratio"]
            })

    return pd.DataFrame(rows)


print("NAVER 검색 API 준비 완료")

In [ ]:
keyword_groups = {
    "파스타": ["파스타"],
    "스테이크": ["스테이크"],
    "브런치": ["브런치"],
    "샐러드": ["샐러드"],
    "피자": ["피자"]
}

test = get_search_interest(
    age_group="20대",
    keyword_groups=keyword_groups
)

test.head()

In [ ]:
age_groups = [
    "20대이하",
    "20대",
    "30대",
    "40대",
    "50대",
    "60대이상"
]

all_age_results = []

for age in age_groups:

    df = get_search_interest(
        age_group=age,
        keyword_groups=keyword_groups
    )

    # 각 메뉴의 기간 평균 검색도
    summary = (
        df.groupby("주제", as_index=False)["검색도"]
        .mean()
    )

    summary["연령대"] = age

    # 해당 연령대 안에서 메뉴 순위
    summary["순위"] = (
        summary["검색도"]
        .rank(
            ascending=False,
            method="min"
        )
        .astype(int)
    )

    all_age_results.append(summary)


age_search_result = pd.concat(
    all_age_results,
    ignore_index=True
)

age_search_result

In [ ]:
rank_table = age_search_result.pivot(
    index="주제",
    columns="연령대",
    values="순위"
)

rank_table = rank_table[
    [
        "20대이하",
        "20대",
        "30대",
        "40대",
        "50대",
        "60대이상"
    ]
]

rank_table

In [ ]:
menu_type = rank_table.copy()

menu_type["전체평균순위"] = (
    menu_type[
        [
            "20대이하",
            "20대",
            "30대",
            "40대",
            "50대",
            "60대이상"
        ]
    ]
    .mean(axis=1)
    .round(2)
)

menu_type["순위표준편차"] = (
    menu_type[
        [
            "20대이하",
            "20대",
            "30대",
            "40대",
            "50대",
            "60대이상"
        ]
    ]
    .std(axis=1)
    .round(2)
)

menu_type["젊은층평균"] = (
    menu_type[
        ["20대이하", "20대"]
    ]
    .mean(axis=1)
    .round(2)
)

menu_type["중년층평균"] = (
    menu_type[
        ["30대", "40대"]
    ]
    .mean(axis=1)
    .round(2)
)

menu_type["고연령평균"] = (
    menu_type[
        ["50대", "60대이상"]
    ]
    .mean(axis=1)
    .round(2)
)

menu_type[
    [
        "전체평균순위",
        "순위표준편차",
        "젊은층평균",
        "중년층평균",
        "고연령평균"
    ]
]

In [ ]:
def classify_menu(row):

    # 전체적으로 검색순위가 낮으면
    if row["전체평균순위"] >= 4:
        return "전반적 약세"

    # 여러 세대에서 고르게 상위권이면
    if (
        row["전체평균순위"] <= 2
        and row["순위표준편차"] <= 1
    ):
        return "세대공통형"

    age_scores = {
        "젊은층형": row["젊은층평균"],
        "중년층형": row["중년층평균"],
        "고연령형": row["고연령평균"]
    }

    # 평균 순위가 가장 낮은 연령군
    return min(
        age_scores,
        key=age_scores.get
    )


menu_type["메뉴유형"] = menu_type.apply(
    classify_menu,
    axis=1
)

menu_type[
    [
        "전체평균순위",
        "순위표준편차",
        "젊은층평균",
        "중년층평균",
        "고연령평균",
        "메뉴유형"
    ]
].sort_values("전체평균순위")

In [ ]:
# 성동구 서양음식 기준
core_age = "30대"
future_age = "60대이상"


# 1. 세대공통 확장 메뉴
common_menu = (
    menu_type[
        menu_type["메뉴유형"] == "세대공통형"
    ]
    .sort_values("전체평균순위")
    .index[0]
)


# 2. 현재 핵심고객 유지 메뉴
# 세대공통 메뉴는 제외하고 현재 핵심연령에서 순위가 가장 높은 메뉴
core_menu = (
    rank_table
    .drop(index=common_menu)
    .sort_values(core_age)
    .index[0]
)


# 3. 미래고객 보완 메뉴
# 앞에서 선택한 메뉴를 제외하고
# 미래 연령에서 가장 순위가 높은 메뉴
future_menu = (
    rank_table
    .drop(index=[common_menu, core_menu])
    .sort_values(future_age)
    .index[0]
)


print("현재 핵심고객 유지 :", core_menu)
print("세대공통 확장     :", common_menu)
print("미래고객 보완     :", future_menu)

In [ ]:
def make_strategy(
    market_type,
    core_menu,
    common_menu,
    future_menu
):

    if market_type == "착시성장형":
        return (
            f"{core_menu}로 기존 핵심고객을 유지하고, "
            f"{common_menu}를 세대공통 확장 메뉴로 활용하며, "
            f"{future_menu}를 통해 미래 고객층을 보완"
        )

    elif market_type == "지속성장형":
        return (
            f"{core_menu}로 현재 성장 기반을 유지하고, "
            f"{common_menu}를 통해 고객층을 넓히며, "
            f"{future_menu}를 미래 성장고객 강화 메뉴로 활용"
        )

    elif market_type == "구조취약형":
        return (
            f"{common_menu}와 같이 여러 세대에서 공통 관심이 있는 "
            f"메뉴를 중심으로 상품구조를 재편하고, "
            f"{future_menu}의 보완 가능성을 추가 검토"
        )

    elif market_type == "회복기회형":
        return (
            f"{future_menu}를 통해 미래 성장고객을 선점하고, "
            f"{common_menu}를 세대공통 확장 메뉴로 활용해 "
            f"고객기반을 조기에 확대"
        )

In [ ]:
strategy = make_strategy(
    market_type="착시성장형",
    core_menu=core_menu,
    common_menu=common_menu,
    future_menu=future_menu
)

print(strategy)

In [ ]:
keyword_groups = {
    "국밥": ["국밥"],
    "찌개": ["찌개"],
    "제육": ["제육"],
    "비빔밥": ["비빔밥"],
    "불고기": ["불고기"]
}

age_groups = [
    "20대이하",
    "20대",
    "30대",
    "40대",
    "50대",
    "60대이상"
]

all_age_results = []

for age in age_groups:

    df = get_search_interest(
        age_group=age,
        keyword_groups=keyword_groups
    )

    summary = (
        df.groupby("주제", as_index=False)["검색도"]
        .mean()
    )

    summary["연령대"] = age

    summary["순위"] = (
        summary["검색도"]
        .rank(
            ascending=False,
            method="min"
        )
        .astype(int)
    )

    all_age_results.append(summary)


pyeongtaek_search = pd.concat(
    all_age_results,
    ignore_index=True
)

pyeongtaek_rank = pyeongtaek_search.pivot(
    index="주제",
    columns="연령대",
    values="순위"
)

pyeongtaek_rank = pyeongtaek_rank[
    [
        "20대이하",
        "20대",
        "30대",
        "40대",
        "50대",
        "60대이상"
    ]
]

pyeongtaek_rank

In [ ]:
def make_menu_type(rank_df):

    result = rank_df.copy()

    age_cols = [
        "20대이하",
        "20대",
        "30대",
        "40대",
        "50대",
        "60대이상"
    ]

    result["전체평균순위"] = (
        result[age_cols]
        .mean(axis=1)
        .round(2)
    )

    result["순위표준편차"] = (
        result[age_cols]
        .std(axis=1)
        .round(2)
    )

    result["젊은층평균"] = (
        result[["20대이하", "20대"]]
        .mean(axis=1)
        .round(2)
    )

    result["중년층평균"] = (
        result[["30대", "40대"]]
        .mean(axis=1)
        .round(2)
    )

    result["고연령평균"] = (
        result[["50대", "60대이상"]]
        .mean(axis=1)
        .round(2)
    )

    def classify(row):

        if row["전체평균순위"] >= 4:
            return "전반적 약세"

        if (
            row["전체평균순위"] <= 2
            and row["순위표준편차"] <= 1
        ):
            return "세대공통형"

        scores = {
            "젊은층형": row["젊은층평균"],
            "중년층형": row["중년층평균"],
            "고연령형": row["고연령평균"]
        }

        return min(scores, key=scores.get)

    result["메뉴유형"] = result.apply(
        classify,
        axis=1
    )

    return result

In [ ]:
pyeongtaek_menu_type = make_menu_type(
    pyeongtaek_rank
)

pyeongtaek_menu_type[
    [
        "전체평균순위",
        "순위표준편차",
        "젊은층평균",
        "중년층평균",
        "고연령평균",
        "메뉴유형"
    ]
].sort_values("전체평균순위")

In [ ]:
keyword_groups = {
    "디저트": ["디저트"],
    "프로틴": ["프로틴"],
    "에너지음료": ["에너지음료"],
    "도시락": ["도시락"],
    "아이스크림": ["아이스크림"]
}

age_groups = [
    "20대이하",
    "20대",
    "30대",
    "40대",
    "50대",
    "60대이상"
]

all_age_results = []

for age in age_groups:

    df = get_search_interest(
        age_group=age,
        keyword_groups=keyword_groups
    )

    summary = (
        df.groupby("주제", as_index=False)["검색도"]
        .mean()
    )

    summary["연령대"] = age

    summary["순위"] = (
        summary["검색도"]
        .rank(
            ascending=False,
            method="min"
        )
        .astype(int)
    )

    all_age_results.append(summary)


sasang_search = pd.concat(
    all_age_results,
    ignore_index=True
)

sasang_rank = sasang_search.pivot(
    index="주제",
    columns="연령대",
    values="순위"
)

sasang_rank = sasang_rank[
    [
        "20대이하",
        "20대",
        "30대",
        "40대",
        "50대",
        "60대이상"
    ]
]

sasang_rank

In [ ]:
sasang_menu_type = make_menu_type(
    sasang_rank
)

sasang_menu_type[
    [
        "전체평균순위",
        "순위표준편차",
        "젊은층평균",
        "중년층평균",
        "고연령평균",
        "메뉴유형"
    ]
].sort_values("전체평균순위")

In [ ]:
def make_menu_type(rank_df):

    result = rank_df.copy()

    age_cols = [
        "20대이하",
        "20대",
        "30대",
        "40대",
        "50대",
        "60대이상"
    ]

    result["전체평균순위"] = (
        result[age_cols].mean(axis=1).round(2)
    )

    result["순위표준편차"] = (
        result[age_cols].std(axis=1).round(2)
    )

    result["젊은층평균"] = (
        result[["20대이하", "20대"]]
        .mean(axis=1)
        .round(2)
    )

    result["중년층평균"] = (
        result[["30대", "40대"]]
        .mean(axis=1)
        .round(2)
    )

    result["고연령평균"] = (
        result[["50대", "60대이상"]]
        .mean(axis=1)
        .round(2)
    )

    def classify(row):

        if row["전체평균순위"] >= 4:
            return "전반적 약세"

        if (
            row["전체평균순위"] <= 2
            and row["순위표준편차"] <= 1
        ):
            return "세대공통형"

        scores = {
            "젊은층": row["젊은층평균"],
            "중년층": row["중년층평균"],
            "고연령": row["고연령평균"]
        }

        best_score = min(scores.values())

        best_groups = [
            group
            for group, score in scores.items()
            if score == best_score
        ]

        if len(best_groups) == 1:
            return best_groups[0] + "형"

        return "복합형(" + "·".join(best_groups) + ")"

    result["메뉴유형"] = result.apply(
        classify,
        axis=1
    )

    return result

In [ ]:
sasang_menu_type = make_menu_type(sasang_rank)

sasang_menu_type[
    [
        "전체평균순위",
        "순위표준편차",
        "젊은층평균",
        "중년층평균",
        "고연령평균",
        "메뉴유형"
    ]
].sort_values("전체평균순위")

In [ ]:
keyword_groups = {
    "밀키트": ["밀키트"],
    "간편식": ["간편식"],
    "건강식품": ["건강식품"],
    "신선식품": ["신선식품"],
    "생필품": ["생필품"]
}

age_groups = [
    "20대이하",
    "20대",
    "30대",
    "40대",
    "50대",
    "60대이상"
]

all_age_results = []

for age in age_groups:

    df = get_search_interest(
        age_group=age,
        keyword_groups=keyword_groups
    )

    summary = (
        df.groupby("주제", as_index=False)["검색도"]
        .mean()
    )

    summary["연령대"] = age

    summary["순위"] = (
        summary["검색도"]
        .rank(
            ascending=False,
            method="min"
        )
        .astype(int)
    )

    all_age_results.append(summary)


siheung_search = pd.concat(
    all_age_results,
    ignore_index=True
)

siheung_rank = siheung_search.pivot(
    index="주제",
    columns="연령대",
    values="순위"
)

siheung_rank = siheung_rank[
    [
        "20대이하",
        "20대",
        "30대",
        "40대",
        "50대",
        "60대이상"
    ]
]

siheung_rank

In [ ]:
siheung_menu_type = make_menu_type(
    siheung_rank
)

siheung_menu_type[
    [
        "전체평균순위",
        "순위표준편차",
        "젊은층평균",
        "중년층평균",
        "고연령평균",
        "메뉴유형"
    ]
].sort_values("전체평균순위")

In [ ]:
strategy_summary = pd.DataFrame([
    {
        "유형": "착시성장형",
        "대표사례": "서울 성동구 서양음식",
        "진단": "현재 성장 / 미래 취약",
        "핵심전략": "방어·다변화",
        "현재고객": "파스타",
        "세대공통": "피자",
        "미래고객": "샐러드",
        "최종해석": "기존 고객을 유지하면서 세대공통 메뉴로 고객기반을 넓히고 미래 고객을 보완"
    },
    {
        "유형": "지속성장형",
        "대표사례": "경기 평택시 일반한식",
        "진단": "현재 성장 / 미래 안정",
        "핵심전략": "유지·확장",
        "현재고객": "국밥·비빔밥",
        "세대공통": "국밥·비빔밥",
        "미래고객": "불고기",
        "최종해석": "세대공통 성장 기반을 유지하면서 미래 성장 고객층까지 확장"
    },
    {
        "유형": "구조취약형",
        "대표사례": "부산 사상구 편의점",
        "진단": "현재 약세 / 미래 취약",
        "핵심전략": "구조재편",
        "현재고객": "디저트·도시락",
        "세대공통": "아이스크림",
        "미래고객": "도시락",
        "최종해석": "세대공통 상품 중심으로 상품구조 재편 가능성을 검토하되 구조적 위험을 함께 경고"
    },
    {
        "유형": "회복기회형",
        "대표사례": "경기 시흥시 대형할인점",
        "진단": "현재 약세 / 미래 안정",
        "핵심전략": "선점",
        "현재고객": "간편식",
        "세대공통": "밀키트",
        "미래고객": "건강식품",
        "최종해석": "세대공통 상품을 먼저 확보하고 미래 성장 고객의 관심 상품을 조기에 강화"
    }
])

strategy_summary

In [ ]:
pizza_age = get_shopping_keyword_age(
    keyword="피자",
    category_code="50000006"
)

pizza_age

In [ ]:
def get_shopping_keyword_age(
    keyword,
    category_code="50000006",
    start_date="2026-01-01",
    end_date="2026-08-31"
):
    url = "https://naverapihub.apigw.ntruss.com/shopping/v1/category/keyword/age"

    data = {
        "startDate": start_date,
        "endDate": end_date,
        "timeUnit": "month",
        "category": category_code,
        "keyword": keyword
    }

    response = requests.post(
        url,
        headers=headers,
        json=data
    )

    if response.status_code != 200:
        print("API 오류:", response.status_code)
        print(response.text)
        return None

    result = response.json()

    df = pd.DataFrame(
        result["results"][0]["data"]
    )

    age_avg = (
        df.groupby("group")["ratio"]
        .mean()
        .reset_index()
    )

    age_avg["연령대"] = age_avg["group"].map({
        "10": "10대",
        "20": "20대",
        "30": "30대",
        "40": "40대",
        "50": "50대",
        "60": "60대이상"
    })

    return age_avg[["연령대", "ratio"]]


print("쇼핑 연령 함수 준비 완료")

In [ ]:
pizza_age = get_shopping_keyword_age(
    keyword="피자",
    category_code="50000006"
)

pizza_age

In [ ]:
shopping_keywords = [
    "파스타",
    "스테이크",
    "브런치",
    "샐러드",
    "피자"
]

shopping_age_results = []

for keyword in shopping_keywords:

    df = get_shopping_keyword_age(
        keyword=keyword,
        category_code="50000006"
    ).copy()

    df["키워드"] = keyword

    shopping_age_results.append(df)


shopping_age_all = pd.concat(
    shopping_age_results,
    ignore_index=True
)

shopping_age_table = shopping_age_all.pivot(
    index="키워드",
    columns="연령대",
    values="ratio"
)

shopping_age_table = shopping_age_table[
    [
        "10대",
        "20대",
        "30대",
        "40대",
        "50대",
        "60대이상"
    ]
].round(1)

shopping_age_table

In [ ]:
interest_peak = (
    shopping_age_table
    .idxmax(axis=1)
    .reset_index()
)

interest_peak.columns = [
    "키워드",
    "관심피크연령"
]

interest_peak

In [ ]:
interest_peak_summary = (
    interest_peak["관심피크연령"]
    .value_counts()
    .reset_index()
)

interest_peak_summary.columns = [
    "관심피크연령",
    "메뉴수"
]

interest_peak_summary

In [ ]:
age_order = [
    "10대",
    "20대",
    "30대",
    "40대",
    "50대",
    "60대이상"
]

# 각 키워드 안에서 연령별 관심순위
shopping_age_rank = shopping_age_table.copy()

shopping_age_rank = shopping_age_rank.rank(
    axis=1,
    ascending=False,
    method="average"
)

# 여러 메뉴의 연령별 평균순위
age_interest_summary = pd.DataFrame({
    "관심평균순위": shopping_age_rank.mean(axis=0)
}).reset_index()

age_interest_summary.columns = [
    "연령대",
    "관심평균순위"
]

age_interest_summary["관심평균순위"] = (
    age_interest_summary["관심평균순위"]
    .round(2)
)

age_interest_summary = (
    age_interest_summary
    .sort_values("관심평균순위")
)

age_interest_summary

In [ ]:
# BC 연령별 소비구조 불러오기
bc_age = pd.read_csv(
    "../data/processed/bc_age_consumption_structure.csv",
    encoding="utf-8-sig"
)

# 서울 성동구 서양음식
seongdong_bc = bc_age[
    (bc_age["시도"] == "서울") &
    (bc_age["시군구"] == "성동구") &
    (bc_age["업종명"] == "서양음식")
][
    ["연령대", "이용금액비중"]
].copy()

seongdong_bc

In [ ]:
# NAVER 관심순위
naver_compare = age_interest_summary[
    age_interest_summary["연령대"].isin(
        ["20대", "30대", "40대", "50대", "60대이상"]
    )
].copy()

# NAVER 순위 다시 계산
naver_compare["관심순위"] = (
    naver_compare["관심평균순위"]
    .rank(method="min")
    .astype(int)
)

# BC도 같은 연령만
bc_compare = seongdong_bc[
    seongdong_bc["연령대"].isin(
        ["20대", "30대", "40대", "50대", "60대이상"]
    )
].copy()

bc_compare["결제순위"] = (
    bc_compare["이용금액비중"]
    .rank(ascending=False, method="min")
    .astype(int)
)

# 결합
age_gap = bc_compare.merge(
    naver_compare[
        ["연령대", "관심평균순위", "관심순위"]
    ],
    on="연령대",
    how="left"
)

# +면 결제순위보다 관심순위가 더 좋음
age_gap["관심-결제 차이"] = (
    age_gap["결제순위"]
    - age_gap["관심순위"]
)

age_gap.sort_values("관심순위")

In [ ]:
print("시도 값:")
print(bc_age["시도"].dropna().unique())

print("\n성동구가 들어간 행:")
display(
    bc_age[
        bc_age["시군구"]
        .astype(str)
        .str.contains("성동", na=False)
    ][
        ["시도", "시군구", "업종명"]
    ]
    .drop_duplicates()
    .head(30)
)

In [ ]:
seongdong_bc = bc_age[
    bc_age["시도"].astype(str).str.contains("서울", na=False) &
    bc_age["시군구"].astype(str).str.contains("성동", na=False) &
    (bc_age["업종명"] == "서양음식")
][
    ["연령대", "이용금액비중"]
].copy()

seongdong_bc

In [ ]:
# 비교 가능한 연령대만 사용
compare_ages = [
    "20대",
    "30대",
    "40대",
    "50대",
    "60대이상"
]

# -----------------------------
# 1. BC 결제 순위
# -----------------------------
bc_compare = seongdong_bc[
    seongdong_bc["연령대"].isin(compare_ages)
].copy()

bc_compare["결제순위"] = (
    bc_compare["이용금액비중"]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)


# -----------------------------
# 2. NAVER 관심 순위
# -----------------------------
naver_compare = age_interest_summary[
    age_interest_summary["연령대"].isin(compare_ages)
].copy()

naver_compare["관심순위"] = (
    naver_compare["관심평균순위"]
    .rank(
        ascending=True,
        method="min"
    )
    .astype(int)
)


# -----------------------------
# 3. 결합
# -----------------------------
age_gap = bc_compare.merge(
    naver_compare[
        [
            "연령대",
            "관심평균순위",
            "관심순위"
        ]
    ],
    on="연령대",
    how="left"
)


# + 값 : 결제에 비해 관심이 더 강함
# - 값 : 관심에 비해 결제가 더 강함
age_gap["관심-결제차이"] = (
    age_gap["결제순위"]
    - age_gap["관심순위"]
)

age_gap = age_gap.sort_values(
    "관심순위"
)

age_gap

In [ ]:
# 각 연령대의 결제순위와 관심순위 차이 절댓값
age_gap["순위차이절댓값"] = (
    age_gap["결제순위"]
    - age_gap["관심순위"]
).abs()

# 평균적으로 몇 순위 정도 차이가 나는지
age_mismatch_score = (
    age_gap["순위차이절댓값"].mean()
)

print(
    "평균 결제-관심 순위 차이 :",
    round(age_mismatch_score, 2),
    "단계"
)

display(
    age_gap[
        [
            "연령대",
            "이용금액비중",
            "결제순위",
            "관심순위",
            "관심-결제차이",
            "순위차이절댓값"
        ]
    ]
)

In [ ]:
def make_age_gap(
    shopping_age_table,
    sido,
    sigungu,
    industry
):
    compare_ages = [
        "20대",
        "30대",
        "40대",
        "50대",
        "60대이상"
    ]

    # -------------------------
    # BC 결제구조
    # -------------------------
    bc_part = bc_age[
        bc_age["시도"].astype(str).str.contains(sido, na=False) &
        bc_age["시군구"].astype(str).str.contains(sigungu, na=False) &
        (bc_age["업종명"] == industry)
    ][
        ["연령대", "이용금액비중"]
    ].copy()

    bc_part = bc_part[
        bc_part["연령대"].isin(compare_ages)
    ]

    bc_part["결제순위"] = (
        bc_part["이용금액비중"]
        .rank(
            ascending=False,
            method="min"
        )
        .astype(int)
    )

    # -------------------------
    # NAVER 관심구조
    # -------------------------
    age_rank = shopping_age_table.rank(
        axis=1,
        ascending=False,
        method="average"
    )

    interest = pd.DataFrame({
        "관심평균순위": age_rank.mean(axis=0)
    }).reset_index()

    interest.columns = [
        "연령대",
        "관심평균순위"
    ]

    interest = interest[
        interest["연령대"].isin(compare_ages)
    ].copy()

    interest["관심순위"] = (
        interest["관심평균순위"]
        .rank(
            ascending=True,
            method="min"
        )
        .astype(int)
    )

    # -------------------------
    # 결합
    # -------------------------
    result = bc_part.merge(
        interest,
        on="연령대",
        how="inner"
    )

    result["관심-결제차이"] = (
        result["결제순위"]
        - result["관심순위"]
    )

    result["순위차이절댓값"] = (
        result["관심-결제차이"].abs()
    )

    mismatch = round(
        result["순위차이절댓값"].mean(),
        2
    )

    return result.sort_values("관심순위"), mismatch

In [ ]:
test_gap, test_score = make_age_gap(
    shopping_age_table,
    sido="서울",
    sigungu="성동",
    industry="서양음식"
)

print("결제-관심 불일치 :", test_score)
display(test_gap)

In [ ]:
pyeongtaek_keywords = [
    "국밥",
    "찌개",
    "제육",
    "비빔밥",
    "불고기"
]

pyeongtaek_shopping_results = []

for keyword in pyeongtaek_keywords:

    df = get_shopping_keyword_age(
        keyword=keyword,
        category_code="50000006"
    ).copy()

    df["키워드"] = keyword
    pyeongtaek_shopping_results.append(df)


pyeongtaek_shopping_all = pd.concat(
    pyeongtaek_shopping_results,
    ignore_index=True
)

pyeongtaek_shopping_table = (
    pyeongtaek_shopping_all
    .pivot(
        index="키워드",
        columns="연령대",
        values="ratio"
    )
)

pyeongtaek_shopping_table = pyeongtaek_shopping_table[
    [
        "10대",
        "20대",
        "30대",
        "40대",
        "50대",
        "60대이상"
    ]
].round(1)

pyeongtaek_shopping_table

In [ ]:
pyeongtaek_gap, pyeongtaek_score = make_age_gap(
    pyeongtaek_shopping_table,
    sido="경기",
    sigungu="평택",
    industry="일반한식"
)

print(
    "평택 일반한식 결제-관심 순위 괴리 :",
    pyeongtaek_score
)

display(pyeongtaek_gap)

In [ ]:
sasang_keywords = [
    "디저트",
    "프로틴",
    "에너지음료",
    "도시락",
    "아이스크림"
]

sasang_shopping_results = []

for keyword in sasang_keywords:

    df = get_shopping_keyword_age(
        keyword=keyword,
        category_code="50000006"
    ).copy()

    df["키워드"] = keyword
    sasang_shopping_results.append(df)


sasang_shopping_all = pd.concat(
    sasang_shopping_results,
    ignore_index=True
)

sasang_shopping_table = (
    sasang_shopping_all
    .pivot(
        index="키워드",
        columns="연령대",
        values="ratio"
    )
)

sasang_shopping_table = sasang_shopping_table[
    [
        "10대",
        "20대",
        "30대",
        "40대",
        "50대",
        "60대이상"
    ]
].round(1)

sasang_shopping_table

In [ ]:
sasang_gap, sasang_score = make_age_gap(
    sasang_shopping_table,
    sido="부산",
    sigungu="사상",
    industry="편의점"
)

print(
    "사상구 편의점 결제-관심 순위 괴리 :",
    sasang_score
)

display(sasang_gap)

In [ ]:
siheung_keywords = [
    "밀키트",
    "간편식",
    "건강식품",
    "신선식품",
    "생필품"
]

siheung_shopping_results = []

for keyword in siheung_keywords:

    df = get_shopping_keyword_age(
        keyword=keyword,
        category_code="50000006"
    ).copy()

    df["키워드"] = keyword
    siheung_shopping_results.append(df)


siheung_shopping_all = pd.concat(
    siheung_shopping_results,
    ignore_index=True
)

siheung_shopping_table = (
    siheung_shopping_all
    .pivot(
        index="키워드",
        columns="연령대",
        values="ratio"
    )
)

siheung_shopping_table = siheung_shopping_table[
    [
        "10대",
        "20대",
        "30대",
        "40대",
        "50대",
        "60대이상"
    ]
].round(1)

siheung_shopping_table

In [ ]:
for keyword in siheung_keywords:
    try:
        df = get_shopping_keyword_age(
            keyword=keyword,
            category_code="50000006"
        )

        print(keyword, "→ 정상")
        display(df)

    except Exception as e:
        print(keyword, "→ 오류 :", e)

In [ ]:
def get_shopping_keyword_age(
    keyword,
    category_code="50000006",
    start_date="2026-01-01",
    end_date="2026-08-31"
):
    url = "https://naverapihub.apigw.ntruss.com/shopping/v1/category/keyword/age"

    data = {
        "startDate": start_date,
        "endDate": end_date,
        "timeUnit": "month",
        "category": category_code,
        "keyword": keyword
    }

    response = requests.post(
        url,
        headers=headers,
        json=data
    )

    if response.status_code != 200:
        print("API 오류:", response.status_code)
        return None

    result = response.json()

    data_rows = result["results"][0]["data"]

    # 데이터가 없으면 중단하지 않고 None 반환
    if len(data_rows) == 0:
        print(f"{keyword} : 연령별 데이터 없음")
        return None

    df = pd.DataFrame(data_rows)

    # group 컬럼이 없는 경우
    if "group" not in df.columns:
        print(
            f"{keyword} : 연령별 group 데이터 없음",
            df.columns.tolist()
        )
        return None

    age_avg = (
        df.groupby("group")["ratio"]
        .mean()
        .reset_index()
    )

    age_avg["연령대"] = age_avg["group"].map({
        "10": "10대",
        "20": "20대",
        "30": "30대",
        "40": "40대",
        "50": "50대",
        "60": "60대이상"
    })

    return age_avg[["연령대", "ratio"]]

In [ ]:
test_frozen = get_shopping_keyword_age(
    keyword="냉동식품",
    category_code="50000006"
)

test_frozen

In [ ]:
siheung_keywords = [
    "밀키트",
    "간편식",
    "건강식품",
    "신선식품",
    "냉동식품"
]

In [ ]:
siheung_shopping_results = []

for keyword in siheung_keywords:

    df = get_shopping_keyword_age(
        keyword=keyword,
        category_code="50000006"
    )

    if df is None:
        print(keyword, "→ 제외")
        continue

    df = df.copy()
    df["키워드"] = keyword

    siheung_shopping_results.append(df)


siheung_shopping_all = pd.concat(
    siheung_shopping_results,
    ignore_index=True
)

siheung_shopping_table = (
    siheung_shopping_all
    .pivot(
        index="키워드",
        columns="연령대",
        values="ratio"
    )
)

# 없는 연령은 NaN으로 그대로 둠
siheung_shopping_table = siheung_shopping_table.reindex(
    columns=[
        "10대",
        "20대",
        "30대",
        "40대",
        "50대",
        "60대이상"
    ]
).round(1)

siheung_shopping_table

In [ ]:
siheung_gap, siheung_score = make_age_gap(
    siheung_shopping_table,
    sido="경기",
    sigungu="시흥",
    industry="대형할인점"
)

print(
    "시흥시 대형할인점 결제-관심 순위 괴리 :",
    siheung_score
)

display(siheung_gap)

In [ ]:
final_strategy = pd.DataFrame([
    {
        "유형": "착시성장형",
        "대표사례": "서울 성동구 서양음식",
        "결제-관심괴리": 1.4,
        "현재고객": "30대",
        "관심고객": "40대",
        "미래고객": "60대이상",
        "세대공통": "피자",
        "보완메뉴상품": "파스타 · 샐러드",
        "전략": "현재고객 방어 + 관심고객 확장 + 미래고객 대비"
    },

    {
        "유형": "지속성장형",
        "대표사례": "경기 평택시 일반한식",
        "결제-관심괴리": 1.0,
        "현재고객": "50대",
        "관심고객": "40대",
        "미래고객": "60대이상",
        "세대공통": "국밥 · 비빔밥",
        "보완메뉴상품": "불고기",
        "전략": "현재 성장기반 유지 + 관심고객 확장 + 미래고객 강화"
    },

    {
        "유형": "구조취약형",
        "대표사례": "부산 사상구 편의점",
        "결제-관심괴리": 1.2,
        "현재고객": "50대",
        "관심고객": "30대",
        "미래고객": "60대이상",
        "세대공통": "아이스크림",
        "보완메뉴상품": "디저트 · 도시락",
        "전략": "관심고객 중심 재편 가능성 검토 + 구조위험 경고"
    },

    {
        "유형": "회복기회형",
        "대표사례": "경기 시흥시 대형할인점",
        "결제-관심괴리": 0.8,
        "현재고객": "40대",
        "관심고객": "30대",
        "미래고객": "60대이상",
        "세대공통": "밀키트",
        "보완메뉴상품": "간편식 · 건강식품",
        "전략": "관심고객 선점 + 미래고객 조기 확보"
    }
])

final_strategy

In [ ]:
final_strategy.to_csv(
    "../data/processed/final_4type_customer_strategy.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")